In [ ]:
# =============================================================================
# MERGE TEST DATASET (Solo terzo anno: 2012-07-01 a 2013-06-30)
# =============================================================================
import sys
import os

# Aggiungi project root al path
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import datetime

# Percorsi file
WX_PATH = "../data/raw/wx_ds_final.xlsx"
PV_PATH = "../data/raw/pv_ds_final.xlsx"
OUTPUT_CSV = "../data/processed/merged_test_ds.csv"
OUTPUT_EXCEL = "../data/processed/merged_test_ds.xlsx"

# 1. Carica SOLO l'ultimo foglio (terzo anno)
xls_wx = pd.ExcelFile(WX_PATH)
xls_pv = pd.ExcelFile(PV_PATH)

print(f"Fogli WX: {xls_wx.sheet_names}")
print(f"Fogli PV: {xls_pv.sheet_names}")

# Leggi solo l'ultimo foglio
last_sheet_wx = xls_wx.sheet_names[-1]
last_sheet_pv = xls_pv.sheet_names[-1]

print(f"\nCaricamento foglio WX: '{last_sheet_wx}'")
print(f"Caricamento foglio PV: '{last_sheet_pv}'")

df_wx = pd.read_excel(xls_wx, sheet_name=last_sheet_wx)
df_pv = pd.read_excel(xls_pv, sheet_name=last_sheet_pv)

print(f"\nWX shape: {df_wx.shape}")
print(f"PV shape: {df_pv.shape}")
print(f"Colonne WX: {df_wx.columns.tolist()}")
print(f"Colonne PV: {df_pv.columns.tolist()}")

# 2. Rinomina colonne PV
df_pv.columns = ["datetime", "pv_power"]

# 3. Merge
df_merged = pd.concat([df_wx, df_pv], axis=1)
df_merged.drop(columns=["datetime"], inplace=True)  # Usa dt_iso, non datetime

# 4. Fix timezone (stessa logica di uploading.py)
df_merged["dt_iso"] = pd.to_datetime(df_merged["dt_iso"], utc=True, errors="raise")
tz_fixed = datetime.timezone(datetime.timedelta(hours=10))  # Sydney +10
df_merged["dt_iso"] = df_merged["dt_iso"].dt.tz_convert(tz_fixed)

print(f"\nDataset merged shape: {df_merged.shape}")
print(f"Date range: {df_merged['dt_iso'].min()} -> {df_merged['dt_iso'].max()}")

# 5. Salva CSV
df_merged.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Salvato: {OUTPUT_CSV}")

# 6. Salva Excel (rimuovi timezone per compatibilità)
df_merged_excel = df_merged.copy()
df_merged_excel["dt_iso"] = df_merged_excel["dt_iso"].dt.tz_localize(None)
df_merged_excel.to_excel(OUTPUT_EXCEL, index=False)
print(f"✅ Salvato: {OUTPUT_EXCEL}")